In [1]:
!git clone https://github.com/bokuwakira1411/llama2.git

Cloning into 'llama2'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 93 (delta 37), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 1.83 MiB | 6.21 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [2]:
%cd llama2

/kaggle/working/llama2


In [3]:
!wget https://www.cs.cmu.edu/~vijayv/stories42M.pt

--2026-04-21 02:57:33--  https://www.cs.cmu.edu/~vijayv/stories42M.pt
Resolving www.cs.cmu.edu (www.cs.cmu.edu)... 128.2.42.95
Connecting to www.cs.cmu.edu (www.cs.cmu.edu)|128.2.42.95|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 232324715 (222M)
Saving to: ‘stories42M.pt’

stories42M.pt       100%[===================>] 221.56M  2.52MB/s    in 68s     

2026-04-21 02:58:42 (3.24 MB/s) - ‘stories42M.pt’ saved [232324715/232324715]



In [ ]:
!python optimizer_test.py

In [ ]:
!python rope_test.py

In [ ]:
!python sanity_check.py

In [4]:
!python  run_llama.py  --option  generate

args: {'train': 'data/cfimdb-train.txt', 'dev': 'data/cfimdb-dev.txt', 'test': 'data/cfimdb-test.txt', 'label_names': 'data/cfimdb-label-mapping.json', 'pretrained_model_path': 'stories42M.pt', 'max_sentence_len': None, 'seed': 1337, 'epochs': 5, 'option': 'generate', 'use_gpu': False, 'generated_sentence_low_temp_out': 'generated-sentence-temp-0.txt', 'generated_sentence_high_temp_out': 'generated-sentence-temp-1.txt', 'dev_out': 'cfimdb-dev-prompting-output.txt', 'test_out': 'cfimdb-test-prompting-output.txt', 'batch_size': 8, 'hidden_dropout_prob': 0.3, 'lr': 2e-05}
load model from stories42M.pt
Temperature is 0.0
I have wanted to see this thriller for a while, and it didn't disappoint. Keanu Reeves, playing the hero John Wick, is this day. He was playing with his toy car, driving it around the living room. Suddenly, he heard a loud crash. He had broken the car and was very sad.
John was angry and he shouted at his little brother. He was only three years old and he was only three. H

In [ ]:
!python  run_llama.py  --option  prompt  --batch_size  10  --train  data/sst-train.txt  --dev  data/sst-dev.txt  --test data/sst-test.txt  --label-names  data/sst-label-mapping.json  --dev_out  sst-dev-prompting-output.txt  --test_out sst-test-prompting-output.txt --use_gpu

In [ ]:
!python run_llama.py --option prompt --batch_size 10 --train data/cfimdb-train.txt --dev data/cfimdb-dev.txt --test data/cfimdb-test.txt --label-names data/cfimdb-label-mapping.json --dev_out cfimdb-dev-prompting-output.txt --test_out cfimdb-test-prompting-output.txt --use_gpu

In [6]:
%%writefile /kaggle/working/llama2/classifier.py
import torch
import torch.nn.functional as F

from config import LlamaConfig
from llama import load_pretrained
from tokenizer import Tokenizer


# =========================
# ZERO-SHOT CLASSIFIER
# =========================
class LlamaZeroShotClassifier(torch.nn.Module):
    def __init__(self, config: LlamaConfig, tokenizer: Tokenizer, label_names: list[str]):
        super().__init__()
        self.num_labels = config.num_labels
        self.llama = load_pretrained(config.pretrained_model_path)

        for param in self.llama.parameters():
            param.requires_grad = False

        assert len(label_names) == self.num_labels
        self.tokenizer = tokenizer
        self.label_name_ids = [
            tokenizer.encode(label, bos=False, eos=False)
            for label in label_names
        ]

    def forward(self, input_ids):
        logits, _ = self.llama(input_ids)
        log_probs = F.log_softmax(logits, dim=-1)

        B, T, V = log_probs.shape
        label_scores = torch.zeros((B, self.num_labels), device=log_probs.device)

        for i, label_token_ids in enumerate(self.label_name_ids):
            score = torch.zeros(B, device=log_probs.device)

            for j, token_id in enumerate(label_token_ids):
                pos = T - len(label_token_ids) + j
                score += log_probs[:, pos, token_id]

            # 🔥 length normalization (important)
            score = score / len(label_token_ids)

            label_scores[:, i] = score

        return label_scores


# =========================
# EMBEDDING CLASSIFIER
# =========================
class LlamaEmbeddingClassifier(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_labels = config.num_labels
        self.llama = load_pretrained(config.pretrained_model_path)

        for param in self.llama.parameters():
            if config.option == 'pretrain':
                param.requires_grad = False
            elif config.option == 'finetune':
                param.requires_grad = True

        dim = self.llama.config.dim

        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)

        self.fc1 = torch.nn.Linear(dim * 2, dim)
        self.fc2 = torch.nn.Linear(dim, dim)
        self.out = torch.nn.Linear(dim, self.num_labels)

        self.activation = torch.nn.GELU()

    def forward(self, input_ids):
        logits, hidden_states = self.llama(input_ids)

        mean_pool = hidden_states.mean(dim=1)
        max_pool, _ = hidden_states.max(dim=1)

        pooled = torch.cat([mean_pool, max_pool], dim=-1)

        x = self.dropout(pooled)

        h = self.activation(self.fc1(x))
        h = self.dropout(h)

        h2 = self.activation(self.fc2(h))
        h = h + h2   

        logits = self.out(h)

        return F.log_softmax(logits, dim=-1)

Overwriting /kaggle/working/llama2/classifier.py


In [7]:
!python run_llama.py --option finetune --epochs 3 --lr 1e-5 --batch_size 50 \
--train data/sst-train.txt \
--dev data/sst-dev.txt \
--test data/sst-test.txt \
--label-names data/sst-label-mapping.json \
--dev_out sst-dev-finetuning-output.txt \
--test_out sst-test-finetuning-output.txt \
--use_gpu

args: {'train': 'data/sst-train.txt', 'dev': 'data/sst-dev.txt', 'test': 'data/sst-test.txt', 'label_names': 'data/sst-label-mapping.json', 'pretrained_model_path': 'stories42M.pt', 'max_sentence_len': None, 'seed': 1337, 'epochs': 3, 'option': 'finetune', 'use_gpu': True, 'generated_sentence_low_temp_out': 'generated-sentence-temp-0.txt', 'generated_sentence_high_temp_out': 'generated-sentence-temp-1.txt', 'dev_out': 'sst-dev-finetuning-output.txt', 'test_out': 'sst-test-finetuning-output.txt', 'batch_size': 50, 'hidden_dropout_prob': 0.3, 'lr': 1e-05}
load 8544 data from data/sst-train.txt
load 1101 data from data/sst-dev.txt
train-0: 100%|████████████████████████████████| 171/171 [00:29<00:00,  5.88it/s]

eval: 100%|███████████████████████████████████| 171/171 [00:10<00:00, 16.28it/s]

eval: 100%|█████████████████████████████████████| 23/23 [00:01<00:00, 17.43it/s]
save the model to finetune-3-1e-05.pt
epoch 0: train loss :: 1.810, train acc :: 0.305, dev acc :: 0.292
train-1: 100%|

In [8]:
%%writefile /kaggle/working/llama2/classifier.py
import torch
import torch.nn.functional as F

from config import LlamaConfig
from llama import load_pretrained
from tokenizer import Tokenizer


class LlamaZeroShotClassifier(torch.nn.Module):
    def __init__(self, config: LlamaConfig, tokenizer: Tokenizer, label_names: list[str]):
        super().__init__()
        self.num_labels = config.num_labels
        self.llama = load_pretrained(config.pretrained_model_path)

        for param in self.llama.parameters():
            param.requires_grad = False

        assert len(label_names) == self.num_labels
        self.tokenizer = tokenizer
        self.label_name_ids = [tokenizer.encode(label, bos=False, eos=False) for label in label_names]

    def forward(self, input_ids):
        logits, _ = self.llama(input_ids)
        log_probs = F.log_softmax(logits, dim=-1)

        B, T, V = log_probs.shape
        label_scores = torch.zeros((B, self.num_labels), device=log_probs.device)

        for i, label_token_ids in enumerate(self.label_name_ids):
            score = torch.zeros(B, device=log_probs.device)

            for j, token_id in enumerate(label_token_ids):
                pos = T - len(label_token_ids) + j
                score += log_probs[:, pos, token_id]

            label_scores[:, i] = score

        return label_scores


class LlamaEmbeddingClassifier(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_labels = config.num_labels
        self.llama = load_pretrained(config.pretrained_model_path)

        for param in self.llama.parameters():
            if config.option == 'pretrain':
                param.requires_grad = False
            elif config.option == 'finetune':
                param.requires_grad = True

        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)

        self.classifier_head = torch.nn.Sequential(
            torch.nn.Linear(self.llama.config.dim, self.llama.config.dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(config.hidden_dropout_prob),
            torch.nn.Linear(self.llama.config.dim, self.num_labels)
        )

    def forward(self, input_ids):
        logits, hidden_states = self.llama(input_ids)

        pooled = hidden_states.mean(dim=1)
        dropped = self.dropout(pooled)

        logits = self.classifier_head(dropped)
        log_probs = F.log_softmax(logits, dim=-1)

        return log_probs

Overwriting /kaggle/working/llama2/classifier.py


In [9]:
!python run_llama.py --option finetune --epochs 5 --lr 1e-5 --batch_size 10 \
--train data/cfimdb-train.txt \
--dev data/cfimdb-dev.txt \
--test data/cfimdb-test.txt \
--label-names data/cfimdb-label-mapping.json \
--dev_out cfimdb-dev-finetuning-output.txt \
--test_out cfimdb-test-finetuning-output.txt \
--use_gpu

args: {'train': 'data/cfimdb-train.txt', 'dev': 'data/cfimdb-dev.txt', 'test': 'data/cfimdb-test.txt', 'label_names': 'data/cfimdb-label-mapping.json', 'pretrained_model_path': 'stories42M.pt', 'max_sentence_len': None, 'seed': 1337, 'epochs': 5, 'option': 'finetune', 'use_gpu': True, 'generated_sentence_low_temp_out': 'generated-sentence-temp-0.txt', 'generated_sentence_high_temp_out': 'generated-sentence-temp-1.txt', 'dev_out': 'cfimdb-dev-finetuning-output.txt', 'test_out': 'cfimdb-test-finetuning-output.txt', 'batch_size': 10, 'hidden_dropout_prob': 0.3, 'lr': 1e-05}
load 1707 data from data/cfimdb-train.txt
load 245 data from data/cfimdb-dev.txt
train-0: 100%|████████████████████████████████| 171/171 [00:45<00:00,  3.76it/s]

eval: 100%|███████████████████████████████████| 171/171 [00:16<00:00, 10.33it/s]

eval: 100%|█████████████████████████████████████| 25/25 [00:02<00:00, 10.32it/s]
save the model to finetune-5-1e-05.pt
epoch 0: train loss :: 0.780, train acc :: 0.597, dev acc 

In [10]:
!python  run_llama.py  --option  prompt  --batch_size  10  --train  data/sst-train.txt  --dev  data/sst-dev.txt  --test data/sst-test.txt  --label-names  data/sst-label-mapping.json  --dev_out  sst-dev-prompting-output.txt  --test_out sst-test-prompting-output.txt --use_gpu

args: {'train': 'data/sst-train.txt', 'dev': 'data/sst-dev.txt', 'test': 'data/sst-test.txt', 'label_names': 'data/sst-label-mapping.json', 'pretrained_model_path': 'stories42M.pt', 'max_sentence_len': None, 'seed': 1337, 'epochs': 5, 'option': 'prompt', 'use_gpu': True, 'generated_sentence_low_temp_out': 'generated-sentence-temp-0.txt', 'generated_sentence_high_temp_out': 'generated-sentence-temp-1.txt', 'dev_out': 'sst-dev-prompting-output.txt', 'test_out': 'sst-test-prompting-output.txt', 'batch_size': 10, 'hidden_dropout_prob': 0.3, 'lr': 2e-05}
load 8544 data from data/sst-train.txt
load 1101 data from data/sst-dev.txt
load 2210 data from data/sst-test.txt
eval: 100%|███████████████████████████████████| 221/221 [00:03<00:00, 67.57it/s]
dev acc :: 0.213
test acc :: 0.224


In [11]:
!python run_llama.py --option prompt --batch_size 10 --train data/cfimdb-train.txt --dev data/cfimdb-dev.txt --test data/cfimdb-test.txt --label-names data/cfimdb-label-mapping.json --dev_out cfimdb-dev-prompting-output.txt --test_out cfimdb-test-prompting-output.txt --use_gpu

args: {'train': 'data/cfimdb-train.txt', 'dev': 'data/cfimdb-dev.txt', 'test': 'data/cfimdb-test.txt', 'label_names': 'data/cfimdb-label-mapping.json', 'pretrained_model_path': 'stories42M.pt', 'max_sentence_len': None, 'seed': 1337, 'epochs': 5, 'option': 'prompt', 'use_gpu': True, 'generated_sentence_low_temp_out': 'generated-sentence-temp-0.txt', 'generated_sentence_high_temp_out': 'generated-sentence-temp-1.txt', 'dev_out': 'cfimdb-dev-prompting-output.txt', 'test_out': 'cfimdb-test-prompting-output.txt', 'batch_size': 10, 'hidden_dropout_prob': 0.3, 'lr': 2e-05}
load 1707 data from data/cfimdb-train.txt
load 245 data from data/cfimdb-dev.txt
load 488 data from data/cfimdb-test.txt
eval: 100%|█████████████████████████████████████| 49/49 [00:04<00:00, 10.81it/s]
dev acc :: 0.502
test acc :: 0.213
